In [441]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from io import StringIO

In [442]:
tickers = [
    "BESI.AS",
    "ALBIO.PA",
    "ENGI.PA",
    "MAU.PA",
    "ALRIB.PA",
    "STMPA.PA",
    "HO.PA",
    "VLA.PA",
    "LQQ.PA",
    "EXA1.AS",
    "ETL.PA",
]

In [443]:
weights = np.array([
    0.0664653,  # BESI
    0.0166811,  # Biosynex
    0.0857736,  # Engie
    0.0352324,  # Maurel & Prom
    0.0288497,  # Riber
    0.0681581,  # STMicro
    0.0887683,  # Thales
    0.0151301,  # Valneva
    0.228948,  # LQQ
    0.339815,  # EXA1
    0.0243977, # Eutelsat
])

In [445]:
prices = yf.download(
    tickers,
    start="2021-12-24",
    end="2026-09-15",
    auto_adjust=True
)["Close"]

[*********************100%***********************]  11 of 11 completed


In [446]:
prices = prices.drop(pd.Timestamp("2026-09-07"))
prices = prices.drop(pd.Timestamp("2025-10-24"))

In [447]:
#print(prices.columns)
print(weights.sum())
print(prices.isna().sum())

0.9982193
Ticker
ALBIO.PA    0
ALRIB.PA    0
BESI.AS     0
ENGI.PA     0
ETL.PA      0
EXA1.AS     0
HO.PA       0
LQQ.PA      0
MAU.PA      0
STMPA.PA    0
VLA.PA      0
dtype: int64


In [448]:
 prices[prices.isna().any(axis=1)] 

Ticker,ALBIO.PA,ALRIB.PA,BESI.AS,ENGI.PA,ETL.PA,EXA1.AS,HO.PA,LQQ.PA,MAU.PA,STMPA.PA,VLA.PA
Date,,,,,,,,,,,


In [449]:
returns = prices.pct_change().dropna()
cov_matrix = returns.cov()
cov_matrix

portfolio_variance = weights.T @ cov_matrix @ weights
portfolio_variance

portfolio_volatility = np.sqrt(portfolio_variance) 
portfolio_volatility

np.float64(0.015316732367251342)

In [450]:
annual_portfolio_volatility = portfolio_volatility * np.sqrt(252)
print(f"{annual_portfolio_volatility * 100} %")

24.314558845128413 %


In [451]:
returns = prices.pct_change(fill_method=None).dropna()
returns.head()

Ticker,ALBIO.PA,ALRIB.PA,BESI.AS,ENGI.PA,ETL.PA,EXA1.AS,HO.PA,LQQ.PA,MAU.PA,STMPA.PA,VLA.PA
Date,,,,,,,,,,,
2021-12-27,0.104273,0.004396,-0.002140,0.004041,0.000465,0.006534,0.007787,0.021554,0.000000,0.009651,0.047657
2021-12-28,-0.038700,-0.015317,0.010185,0.015789,-0.000930,0.006607,0.006395,0.001871,0.031603,0.002276,-0.025777
2021-12-29,-0.023349,-0.007778,-0.021757,0.001829,-0.000466,-0.001385,-0.004766,-0.011827,-0.019694,-0.005563,-0.029572
2021-12-30,0.083265,-0.016797,0.013019,-0.005780,-0.000466,0.000494,0.001596,0.013858,0.000000,-0.001256,-0.010425
2021-12-31,0.001522,-0.003417,0.004284,-0.004437,0.000466,-0.001747,-0.006640,-0.013876,0.017857,-0.008459,-0.007293


In [452]:
cov_matrix

Ticker,ALBIO.PA,ALRIB.PA,BESI.AS,ENGI.PA,ETL.PA,EXA1.AS,HO.PA,LQQ.PA,MAU.PA,STMPA.PA,VLA.PA
Ticker,,,,,,,,,,,
ALBIO.PA,0.002321,0.000037,0.000104,0.000058,0.000136,0.000049,0.000025,0.000111,0.000019,0.000027,0.000205
ALRIB.PA,0.000037,0.001079,0.000197,0.000038,0.000142,0.000107,0.000050,0.000207,0.000042,0.000200,0.000140
BESI.AS,0.000104,0.000197,0.000964,0.000052,0.000118,0.000157,0.000019,0.000456,0.000108,0.000474,0.000191
ENGI.PA,0.000058,0.000038,0.000052,0.000190,0.000019,0.000082,0.000030,0.000046,0.000021,0.000057,0.000063
ETL.PA,0.000136,0.000142,0.000118,0.000019,0.003638,0.000103,0.000235,0.000050,-0.000007,0.000091,0.000124
EXA1.AS,0.000049,0.000107,0.000157,0.000082,0.000103,0.000250,0.000036,0.000177,0.000041,0.000167,0.000176
HO.PA,0.000025,0.000050,0.000019,0.000030,0.000235,0.000036,0.000335,0.000056,0.000077,0.000009,0.000020
LQQ.PA,0.000111,0.000207,0.000456,0.000046,0.000050,0.000177,0.000056,0.000658,0.000094,0.000386,0.000277
MAU.PA,0.000019,0.000042,0.000108,0.000021,-0.000007,0.000041,0.000077,0.000094,0.000758,0.000068,0.000115


In [453]:
print(len(tickers))
print(len(weights))
print(weights.sum())

11
11
0.9982193


In [454]:
portfolio_returns = returns @ weights
portfolio_returns.head()

Date
2021-12-27    0.012881
2021-12-28    0.007001
2021-12-29   -0.011314
2021-12-30    0.005857
2021-12-31    0.000386
dtype: float64

In [456]:
portfolio_volatility_direct = portfolio_returns.std()
portfolio_volatility_direct

np.float64(0.015316732367251342)

In [457]:
np.sqrt(portfolio_variance)
# portfolio_variance et portfolio_volatility_direct sont identiques, parfait!

np.float64(0.015316732367251342)

In [458]:
candidate = yf.download(
    "SAN.PA",
    start="2021-12-24",
    end="2026-09-15",
    auto_adjust = True
)["Close"]
candidate.head()

[*********************100%***********************]  1 of 1 completed


Ticker,SAN.PA
Date,
2021-12-24,70.910995
2021-12-27,71.484032
2021-12-28,71.734230
2021-12-29,71.669662
2021-12-30,71.936005


In [459]:
candidate_returns = candidate.pct_change(fill_method=None).dropna()
candidate_returns.head()

Ticker,SAN.PA
Date,
2021-12-27,0.008081
2021-12-28,0.003500
2021-12-29,-0.000900
2021-12-30,0.003716
2021-12-31,-0.006171


In [460]:
comparison = pd.concat(
    [candidate_returns, portfolio_returns],
    axis=1,
    join="inner"
)
comparison.columns = ["Mon portefeuille", "Sanofi"]

comparison.head()

,Mon portefeuille,Sanofi
Date,,
2021-12-27,0.008081,0.012881
2021-12-28,0.003500,0.007001
2021-12-29,-0.000900,-0.011314
2021-12-30,0.003716,0.005857
2021-12-31,-0.006171,0.000386


In [461]:
comparison.shape

(1205, 2)

In [462]:
correlation_de_Pearson = comparison["Mon portefeuille"].corr(
    comparison["Sanofi"],
    method="pearson"
)
correlation_de_Pearson

np.float64(0.07521474869074302)

In [463]:
candidates_113 = [
    # Les 100 nouvelles
    "OR.PA",
    "MC.PA",
    "RMS.PA",
    "BNP.PA",
    "CS.PA",
    "CDI.PA",
    "EL.PA",
    "DG.PA",
    "BN.PA",
    "LR.PA",
    "SGO.PA",
    "KER.PA",
    "DSY.PA",
    "PUB.PA",
    "VIE.PA",
    "ML.PA",
    "AM.PA",
    "CAP.PA",
    "AMUN.PA",
    "EN.PA",
    "RI.PA",
    "URW.PA",
    "IPN.PA",
    "ERF.PA",
    "BVI.PA",
    "CA.PA",
    "ADP.PA",
    "LI.PA",
    "AC.PA",
    "RXL.PA",
    "BOL.PA",
    "FGR.PA",
    "GET.PA",
    "BIM.PA",
    "SW.PA",
    "AYV.PA",
    "ABVX.PA",
    "SPIE.PA",
    "ALO.PA",
    "EDEN.PA",
    "SCR.PA",
    "NEX.PA",
    "ODET.PA",
    "COV.PA",
    "DEC.PA",
    "ELIS.PA",
    "GFC.PA",
    "SOI.PA",
    "AKE.PA",
    "FDJU.PA",
    "TEP.PA",
    "LOUP.PA",
    "RUI.PA",
    "FR.PA",
    "MF.PA",
    "SOP.PA",
    "SK.PA",
    "CAF.PA",
    "AF.PA",
    "RF.PA",
    "FII.PA",
    "TRI.PA",
    "TKO.PA",
    "EXENS.PA",
    "BB.PA",
    "VCT.PA",
    "VIRP.PA",
    "MMB.PA",
    "ITP.PA",
    "ATE.PA",
    "ARTO.PA",
    "COFA.PA",
    "VRLA.PA",
    "OVH.PA",
    "RCO.PA",
    "FMONC.PA",
    "PLX.PA",
    "CARM.PA",
    "IDL.PA",
    "ALTA.PA",
    "EMEIS.PA",
    "NK.PA",
    "ARG.PA",
    "FRVIA.PA",
    "OPM.PA",
    "CBE.PA",
    "STF.PA",
    "VIV.PA",
    "MMT.PA",
    "IPS.PA",
    "DBG.PA",
    "TFI.PA",
    "CLARI.PA",
    "ANTIN.PA",
    "PEUG.PA",
    "ICAD.PA",
    "ERA.PA",
    "LSS.PA",
    "GLO.PA",
    "WLN.PA",
    "MEDCL.PA",
    "UBI.PA",
    "WAVE.PA",
    "74SW.PA",
    "BSD.PA",

    # Les 13 déjà testées
    "NAE.PA",
    "ATO.PA",
    "DBV.PA",
    "VK.PA",
    "NANO.PA",
    "GTT.PA",
    "BAIN.PA",
    "SU.PA",
    "AI.PA",
    "AIR.PA",
    "ORA.PA",
    "SAF.PA",
    "GLE.PA",
]

In [464]:
candidates2 = yf.download(
    candidates_113,
    start="2021-12-24",
    end="2026-09-15",
    auto_adjust = True
)["Close"]
candidates2.head

[*********************100%***********************]  118 of 118 completed


<bound method NDFrame.head of Ticker        74SW.PA     ABVX.PA      AC.PA      ADP.PA      AF.PA  \
Date                                                                  
2021-12-24  25.329430   27.799999  25.071440  100.575096  19.881388   
2021-12-27  25.618359   27.799999  25.302267   99.591263  19.841040   
2021-12-28  26.196217   27.950001  25.240122  100.127892  19.866257   
2021-12-29  26.196217   28.650000  25.124708  100.172623  19.619125   
2021-12-30  26.003595   28.650000  25.373291  100.262054  19.548517   
...               ...         ...        ...         ...        ...   
2026-09-08  38.299999  101.000000  47.000000  111.500000  11.780000   
2026-09-09  37.799999  101.199997  46.310001  108.800003  11.585000   
2026-09-10  37.700001  100.300003  45.799999  108.300003  11.585000   
2026-09-11  38.000000  101.300003  46.070000  108.900002  11.635000   
2026-09-14  39.000000   99.650002  45.730000  108.900002  11.420000   

Ticker           AI.PA      AIR.PA      AKE.PA

In [465]:
returns_candidates_113 = candidates2.pct_change(fill_method=None).dropna()
returns_candidates_113.head()

Ticker,74SW.PA,ABVX.PA,AC.PA,ADP.PA,AF.PA,AI.PA,AIR.PA,AKE.PA,ALO.PA,ALTA.PA,...,UBI.PA,URW.PA,VCT.PA,VIE.PA,VIRP.PA,VIV.PA,VK.PA,VRLA.PA,WAVE.PA,WLN.PA
Date,,,,,,,,,,,,,,,,,,,,,
2024-06-10,0.000000,0.007752,-0.003337,-0.040507,-0.018182,-0.012537,-0.003600,-0.005025,-0.000848,-0.014706,...,-0.006443,-0.010834,-0.016238,-0.017453,-0.005510,-0.009865,0.006231,-0.009504,-0.034226,-0.031713
2024-06-11,0.008000,0.006154,-0.006181,-0.019560,-0.020468,-0.013205,-0.006423,0.000561,-0.050325,-0.075605,...,-0.039775,-0.041009,-0.009629,-0.024671,-0.020776,-0.009963,-0.024458,-0.004264,-0.033898,-0.023820
2024-06-12,-0.007937,0.001529,0.016066,0.002494,0.027861,0.020012,0.005791,-0.003926,-0.000893,-0.032715,...,-0.009905,0.015405,0.005556,0.007757,-0.001414,0.012323,-0.005712,0.017131,0.011164,0.002614
2024-06-13,0.008000,-0.016794,-0.021678,-0.018242,-0.028558,-0.035102,-0.023835,-0.030968,-0.044398,-0.007892,...,-0.007731,-0.010986,-0.006906,-0.020080,-0.028329,-0.005681,-0.038621,-0.024210,-0.042587,-0.058236
2024-06-14,-0.007937,-0.080745,-0.034672,-0.047297,-0.034180,-0.024498,-0.013580,-0.027310,-0.033053,-0.038636,...,-0.020165,-0.041259,-0.037552,-0.046448,-0.010204,-0.030810,-0.038845,-0.018878,-0.023064,-0.044762


In [466]:
comparison = pd.concat(
    [portfolio_returns, returns_candidates_113 ],
    axis=1,
    join="inner"
)
comparison.columns = [
    "Mon portefeuille", 
    "OR.PA",
    "MC.PA",
    "RMS.PA",
    "BNP.PA",
    "CS.PA",
    "CDI.PA",
    "EL.PA",
    "DG.PA",
    "BN.PA",
    "LR.PA",
    "SGO.PA",
    "KER.PA",
    "DSY.PA",
    "PUB.PA",
    "VIE.PA",
    "ML.PA",
    "AM.PA",
    "CAP.PA",
    "AMUN.PA",
    "EN.PA",
    "RI.PA",
    "URW.PA",
    "IPN.PA",
    "ERF.PA",
    "BVI.PA",
    "CA.PA",
    "ADP.PA",
    "LI.PA",
    "AC.PA",
    "RXL.PA",
    "BOL.PA",
    "FGR.PA",
    "GET.PA",
    "BIM.PA",
    "SW.PA",
    "AYV.PA",
    "ABVX.PA",
    "SPIE.PA",
    "ALO.PA",
    "EDEN.PA",
    "SCR.PA",
    "NEX.PA",
    "ODET.PA",
    "COV.PA",
    "DEC.PA",
    "ELIS.PA",
    "GFC.PA",
    "SOI.PA",
    "AKE.PA",
    "FDJU.PA",
    "TEP.PA",
    "LOUP.PA",
    "RUI.PA",
    "FR.PA",
    "MF.PA",
    "SOP.PA",
    "SK.PA",
    "CAF.PA",
    "AF.PA",
    "RF.PA",
    "FII.PA",
    "TRI.PA",
    "TKO.PA",
    "EXENS.PA",
    "BB.PA",
    "VCT.PA",
    "VIRP.PA",
    "MMB.PA",
    "ITP.PA",
    "ATE.PA",
    "ARTO.PA",
    "COFA.PA",
    "VRLA.PA",
    "OVH.PA",
    "RCO.PA",
    "FMONC.PA",
    "PLX.PA",
    "CARM.PA",
    "IDL.PA",
    "ALTA.PA",
    "EMEIS.PA",
    "NK.PA",
    "ARG.PA",
    "FRVIA.PA",
    "OPM.PA",
    "CBE.PA",
    "STF.PA",
    "VIV.PA",
    "MMT.PA",
    "IPS.PA",
    "DBG.PA",
    "TFI.PA",
    "CLARI.PA",
    "ANTIN.PA",
    "PEUG.PA",
    "ICAD.PA",
    "ERA.PA",
    "LSS.PA",
    "GLO.PA",
    "WLN.PA",
    "MEDCL.PA",
    "UBI.PA",
    "WAVE.PA",
    "74SW.PA",
    "BSD.PA",

    # Les 13 déjà testées
    "NAE.PA",
    "ATO.PA",
    "DBV.PA",
    "VK.PA",
    "NANO.PA",
    "GTT.PA",
    "BAIN.PA",
    "SU.PA",
    "AI.PA",
    "AIR.PA",
    "ORA.PA",
    "SAF.PA",
    "GLE.PA",
]
comparison.head()

,Mon portefeuille,OR.PA,MC.PA,RMS.PA,BNP.PA,CS.PA,CDI.PA,EL.PA,DG.PA,BN.PA,...,VK.PA,NANO.PA,GTT.PA,BAIN.PA,SU.PA,AI.PA,AIR.PA,ORA.PA,SAF.PA,GLE.PA
Date,,,,,,,,,,,,,,,,,,,,,
2024-06-10,-0.011534,0.000000,0.007752,-0.003337,-0.040507,-0.018182,-0.012537,-0.003600,-0.005025,-0.000848,...,-0.006443,-0.010834,-0.016238,-0.017453,-0.005510,-0.009865,0.006231,-0.009504,-0.034226,-0.031713
2024-06-11,-0.007235,0.008000,0.006154,-0.006181,-0.019560,-0.020468,-0.013205,-0.006423,0.000561,-0.050325,...,-0.039775,-0.041009,-0.009629,-0.024671,-0.020776,-0.009963,-0.024458,-0.004264,-0.033898,-0.023820
2024-06-12,0.016047,-0.007937,0.001529,0.016066,0.002494,0.027861,0.020012,0.005791,-0.003926,-0.000893,...,-0.009905,0.015405,0.005556,0.007757,-0.001414,0.012323,-0.005712,0.017131,0.011164,0.002614
2024-06-13,-0.023052,0.008000,-0.016794,-0.021678,-0.018242,-0.028558,-0.035102,-0.023835,-0.030968,-0.044398,...,-0.007731,-0.010986,-0.006906,-0.020080,-0.028329,-0.005681,-0.038621,-0.024210,-0.042587,-0.058236
2024-06-14,-0.034701,-0.007937,-0.080745,-0.034672,-0.047297,-0.034180,-0.024498,-0.013580,-0.027310,-0.033053,...,-0.020165,-0.041259,-0.037552,-0.046448,-0.010204,-0.030810,-0.038845,-0.018878,-0.023064,-0.044762


In [467]:
comparison.shape

(577, 119)

In [468]:
correlation_de_Pearson2 = comparison[
    [
    "OR.PA",
    "MC.PA",
    "RMS.PA",
    "BNP.PA",
    "CS.PA",
    "CDI.PA",
    "EL.PA",
    "DG.PA",
    "BN.PA",
    "LR.PA",
    "SGO.PA",
    "KER.PA",
    "DSY.PA",
    "PUB.PA",
    "VIE.PA",
    "ML.PA",
    "AM.PA",
    "CAP.PA",
    "AMUN.PA",
    "EN.PA",
    "RI.PA",
    "URW.PA",
    "IPN.PA",
    "ERF.PA",
    "BVI.PA",
    "CA.PA",
    "ADP.PA",
    "LI.PA",
    "AC.PA",
    "RXL.PA",
    "BOL.PA",
    "FGR.PA",
    "GET.PA",
    "BIM.PA",
    "SW.PA",
    "AYV.PA",
    "ABVX.PA",
    "SPIE.PA",
    "ALO.PA",
    "EDEN.PA",
    "SCR.PA",
    "NEX.PA",
    "ODET.PA",
    "COV.PA",
    "DEC.PA",
    "ELIS.PA",
    "GFC.PA",
    "SOI.PA",
    "AKE.PA",
    "FDJU.PA",
    "TEP.PA",
    "LOUP.PA",
    "RUI.PA",
    "FR.PA",
    "MF.PA",
    "SOP.PA",
    "SK.PA",
    "CAF.PA",
    "AF.PA",
    "RF.PA",
    "FII.PA",
    "TRI.PA",
    "TKO.PA",
    "EXENS.PA",
    "BB.PA",
    "VCT.PA",
    "VIRP.PA",
    "MMB.PA",
    "ITP.PA",
    "ATE.PA",
    "ARTO.PA",
    "COFA.PA",
    "VRLA.PA",
    "OVH.PA",
    "RCO.PA",
    "FMONC.PA",
    "PLX.PA",
    "CARM.PA",
    "IDL.PA",
    "ALTA.PA",
    "EMEIS.PA",
    "NK.PA",
    "ARG.PA",
    "FRVIA.PA",
    "OPM.PA",
    "CBE.PA",
    "STF.PA",
    "VIV.PA",
    "MMT.PA",
    "IPS.PA",
    "DBG.PA",
    "TFI.PA",
    "CLARI.PA",
    "ANTIN.PA",
    "PEUG.PA",
    "ICAD.PA",
    "ERA.PA",
    "LSS.PA",
    "GLO.PA",
    "WLN.PA",
    "MEDCL.PA",
    "UBI.PA",
    "WAVE.PA",
    "74SW.PA",
    "BSD.PA",

    # Les 13 déjà testées
    "NAE.PA",
    "ATO.PA",
    "DBV.PA",
    "VK.PA",
    "NANO.PA",
    "GTT.PA",
    "BAIN.PA",
    "SU.PA",
    "AI.PA",
    "AIR.PA",
    "ORA.PA",
    "SAF.PA",
    "GLE.PA",
 ]
].corrwith(
    comparison["Mon portefeuille"],
    method="pearson"
)

correlation_de_Pearson2

OR.PA     0.082495
MC.PA    -0.005863
RMS.PA    0.276155
BNP.PA    0.151509
CS.PA     0.170781
            ...   
AI.PA     0.054170
AIR.PA    0.375021
ORA.PA    0.228192
SAF.PA    0.181253
GLE.PA    0.193363
Length: 118, dtype: float64

In [469]:
top_10 = correlation_de_Pearson2.nlargest(10)
bottom_10 = correlation_de_Pearson2.nsmallest(10)

print(f"voici le top 10: {top_10}")
print("--------------------------------------")
print(f"voici le bottom 10: {bottom_10}")

voici le top 10: WAVE.PA     0.511105
ANTIN.PA    0.472314
ITP.PA      0.451491
GLO.PA      0.404563
KER.PA      0.392901
IPS.PA      0.391314
GFC.PA      0.375890
MF.PA       0.375775
AIR.PA      0.375021
CBE.PA      0.375009
dtype: float64
--------------------------------------
voici le bottom 10: URW.PA     -0.128821
FRVIA.PA   -0.055214
RI.PA      -0.051436
AM.PA      -0.014304
MC.PA      -0.005863
VIE.PA     -0.002743
AI.PA       0.054170
ADP.PA      0.056430
VIRP.PA     0.064587
DBG.PA      0.079876
dtype: float64


In [470]:
correlation_20 = pd.concat([top_10, bottom_10])
correlation_20

WAVE.PA     0.511105
ANTIN.PA    0.472314
ITP.PA      0.451491
GLO.PA      0.404563
KER.PA      0.392901
IPS.PA      0.391314
GFC.PA      0.375890
MF.PA       0.375775
AIR.PA      0.375021
CBE.PA      0.375009
URW.PA     -0.128821
FRVIA.PA   -0.055214
RI.PA      -0.051436
AM.PA      -0.014304
MC.PA      -0.005863
VIE.PA     -0.002743
AI.PA       0.054170
ADP.PA      0.056430
VIRP.PA     0.064587
DBG.PA      0.079876
dtype: float64

In [471]:
tickers_20 = correlation_20.index.tolist()

In [472]:
#sur ces actions (les plus correlés et les moins voire décorelés pour les negatives), on va voir lesquelles sont interéssantes, d'abord en annualisant les rendements
prices_20 = yf.download(
    tickers_20,
    start="2021-12-24",
    end="2026-09-15",
    auto_adjust=True
)["Close"]
prices_20

[*********************100%***********************]  20 of 20 completed


Ticker,ADP.PA,AI.PA,AIR.PA,AM.PA,ANTIN.PA,CBE.PA,DBG.PA,FRVIA.PA,GFC.PA,GLO.PA,IPS.PA,ITP.PA,KER.PA,MC.PA,MF.PA,RI.PA,URW.PA,VIE.PA,VIRP.PA,WAVE.PA
Date,,,,,,,,,,,,,,,,,,,,
2021-12-24,100.575096,103.053795,102.459152,87.019409,26.827648,916.000000,8.257899,35.943520,89.491928,15.780991,33.732346,40.234295,600.915344,653.961182,84.296158,176.195251,NaN,25.703440,406.833191,49.945221
2021-12-27,99.591255,104.368675,103.064651,86.789078,27.193192,916.000000,8.515155,36.180332,89.934959,15.055632,33.690651,40.461285,611.411621,662.074036,85.491287,175.777924,NaN,25.598497,415.645538,49.179779
2021-12-28,100.127899,105.108315,104.055489,87.157616,26.938902,916.000000,8.480855,36.311901,90.414894,15.020248,33.857433,40.574780,612.811157,666.176086,84.614845,176.111755,NaN,26.300823,416.135071,49.945221
2021-12-29,100.172615,104.601532,102.862831,86.881218,26.875324,916.000000,8.562318,35.680386,90.304153,14.878714,34.149311,41.085514,615.347839,663.532654,84.455513,176.779495,NaN,26.026350,422.989075,50.519310
2021-12-30,100.262054,105.245277,103.376587,87.157616,27.463375,916.000000,8.558029,36.118938,90.931770,15.214858,34.649666,41.709740,619.108887,665.446838,84.614845,176.946426,NaN,25.897186,421.030762,51.284744
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-08,111.500000,173.160004,200.699997,281.399994,8.420000,676.000000,9.215000,9.880000,67.449997,27.350000,36.060001,28.000000,244.399994,426.549988,86.150002,60.720001,94.820000,31.950001,316.000000,31.100000
2026-09-09,108.800003,168.320007,196.339996,278.799988,8.300000,672.000000,9.435000,9.520000,66.349998,26.600000,35.340000,28.160000,232.250000,411.250000,85.199997,60.040001,93.839996,31.559999,308.000000,30.049999
2026-09-10,108.300003,166.779999,195.779999,279.600006,8.160000,669.000000,9.370000,9.732000,65.750000,26.450001,35.900002,27.700001,234.149994,406.500000,83.750000,59.500000,93.019997,31.440001,302.000000,29.500000


In [689]:
tickers_20 = [ticker for ticker in tickers_20 if ticker != "URW.PA"]

prices_20 = prices_20.drop(columns="URW.PA", errors="ignore")

candidates_returns = prices_20.pct_change(fill_method=None).dropna()
print(pd.Timestamp("2026-09-07") in prices_20.index)
print(pd.Timestamp("2026-09-07") in candidates_returns.index)

True
True


In [690]:
print("URW.PA" in prices_20.columns)

False


In [691]:
print("URW.PA" in candidates_returns.columns)

False


In [692]:
print(candidates_returns["GLO.PA"].isna().any())

False


In [693]:
candidates_returns.isna().sum()

Ticker
ADP.PA      0
AI.PA       0
AIR.PA      0
AM.PA       0
ANTIN.PA    0
CBE.PA      0
DBG.PA      0
FRVIA.PA    0
GFC.PA      0
GLO.PA      0
IPS.PA      0
ITP.PA      0
KER.PA      0
MC.PA       0
MF.PA       0
RI.PA       0
VIE.PA      0
VIRP.PA     0
WAVE.PA     0
dtype: int64

In [694]:
print("URW.PA" in tickers_20) 
print(candidates_returns.describe())

False
Ticker       ADP.PA        AI.PA       AIR.PA        AM.PA     ANTIN.PA  \
count   1207.000000  1207.000000  1207.000000  1207.000000  1207.000000   
mean       0.000215     0.000472     0.000691     0.001148    -0.000701   
std        0.017180     0.012101     0.017738     0.018486     0.024559   
min       -0.125973    -0.072748    -0.094099    -0.090490    -0.134957   
25%       -0.008891    -0.006050    -0.008482    -0.008096    -0.014982   
50%        0.000000     0.000582     0.001228     0.001029     0.000000   
75%        0.009454     0.007087     0.009907     0.010358     0.013901   
max        0.096257     0.082596     0.095719     0.147727     0.121778   

Ticker       CBE.PA       DBG.PA     FRVIA.PA       GFC.PA       GLO.PA  \
count   1207.000000  1207.000000  1207.000000  1207.000000  1207.000000   
mean      -0.000037     0.000380    -0.000498    -0.000144     0.000655   
std        0.021736     0.023400     0.034936     0.014939     0.020807   
min       -0.18626

In [695]:
#on va vérifier si il y a des infinis (pour checker si il y a des soucis), puis calculer le rendement total des candidats
#et prendre leur rendement annualisé (que l'on utilisera par la suite)

print(np.isinf(candidates_returns).any())

print(candidates_returns.max())

rendement_total = (1 + candidates_returns).prod() - 1

print(rendement_total)

rendement_annualisé = ((1 + rendement_total) ** (1/5)) - 1

print(rendement_annualisé)

Ticker
ADP.PA      False
AI.PA       False
AIR.PA      False
AM.PA       False
ANTIN.PA    False
CBE.PA      False
DBG.PA      False
FRVIA.PA    False
GFC.PA      False
GLO.PA      False
IPS.PA      False
ITP.PA      False
KER.PA      False
MC.PA       False
MF.PA       False
RI.PA       False
VIE.PA      False
VIRP.PA     False
WAVE.PA     False
dtype: bool
Ticker
ADP.PA      0.096257
AI.PA       0.082596
AIR.PA      0.095719
AM.PA       0.147727
ANTIN.PA    0.121778
CBE.PA      0.123784
DBG.PA      0.164427
FRVIA.PA    0.172475
GFC.PA      0.067531
GLO.PA      0.121776
IPS.PA      0.082925
ITP.PA      0.197689
KER.PA      0.169062
MC.PA       0.128119
MF.PA       0.090461
RI.PA       0.079413
VIE.PA      0.122616
VIRP.PA     0.120066
WAVE.PA     0.139676
dtype: float64
Ticker
ADP.PA      0.082773
AI.PA       0.619348
AIR.PA      0.903197
AM.PA       2.254447
ANTIN.PA   -0.702546
CBE.PA     -0.281550
DBG.PA      0.136488
FRVIA.PA   -0.738812
GFC.PA     -0.265856
GLO.PA      0.701414
I

In [696]:
print("URW.PA" in candidates_returns) 
candidates_returns.shape

False


(1207, 19)

In [697]:
candidates_returns.isna().sum()

Ticker
ADP.PA      0
AI.PA       0
AIR.PA      0
AM.PA       0
ANTIN.PA    0
CBE.PA      0
DBG.PA      0
FRVIA.PA    0
GFC.PA      0
GLO.PA      0
IPS.PA      0
ITP.PA      0
KER.PA      0
MC.PA       0
MF.PA       0
RI.PA       0
VIE.PA      0
VIRP.PA     0
WAVE.PA     0
dtype: int64

In [698]:
#on passe au calcul de la vol de chaque actions

vol_quotidienne_candidates_returns = candidates_returns.std()

vol_quotidienne_candidates_returns

Ticker
ADP.PA      0.017180
AI.PA       0.012101
AIR.PA      0.017738
AM.PA       0.018486
ANTIN.PA    0.024559
CBE.PA      0.021736
DBG.PA      0.023400
FRVIA.PA    0.034936
GFC.PA      0.014939
GLO.PA      0.020807
IPS.PA      0.019034
ITP.PA      0.020203
KER.PA      0.023150
MC.PA       0.018976
MF.PA       0.014970
RI.PA       0.016024
VIE.PA      0.015260
VIRP.PA     0.019869
WAVE.PA     0.024387
dtype: float64

In [699]:
#on va chercher la volatilité annualisé des 19 actions afin de déterminer l'impact du risque de chacune dans le pf test
vol_annualisé_candidates_returns = vol_quotidienne_candidates_returns * np.sqrt(252)
print(f"Voici la liste des vol annualisées des 19 actions :\n{vol_annualisé_candidates_returns.mul(100).map(lambda x: f'{x:.2f}%')}")

Voici la liste des vol annualisées des 19 actions :
Ticker
ADP.PA      27.27%
AI.PA       19.21%
AIR.PA      28.16%
AM.PA       29.35%
ANTIN.PA    38.99%
CBE.PA      34.51%
DBG.PA      37.15%
FRVIA.PA    55.46%
GFC.PA      23.71%
GLO.PA      33.03%
IPS.PA      30.22%
ITP.PA      32.07%
KER.PA      36.75%
MC.PA       30.12%
MF.PA       23.76%
RI.PA       25.44%
VIE.PA      24.22%
VIRP.PA     31.54%
WAVE.PA     38.71%
dtype: str


In [700]:
#On passe au sharpe ratio. Il permet, en faisant la difference du rendement annualisé par rapport au taux sans risque
#sur la vol annualisé. Il mesure donc le rendement par unité de risque en prenant en compte un certain cout d'opportunité

#problème méthodo: on prends quel taux sans risque? étant donné que ce sont des actions françaises, on prendra l'€STR, et non 
#des bons du trésor us à 1 ou 3 mois. Deuxième problème: le taux évolue. On va alors récupérer les taux grâce à lAPI de la BCE via requests
#en les prenant aux mêmes dates que celles qu'on a choisi auparavant

start_estr = "2021-12-24"
end_estr = "2026-09-15"

url_estr = f"https://data-api.ecb.europa.eu/service/data/EST/B.EU000A2X2A25.WT?startPeriod={start_estr}&endPeriod={end_estr}&format=csvdata"

response_estr = requests.get(url_estr)
response_estr.status_code

#status code récuperé = 200: gagné

response_estr.text[:500]

'KEY,FREQ,BENCHMARK_ITEM,DATA_TYPE_EST,TIME_PERIOD,OBS_VALUE,OBS_STATUS,CONF_STATUS,PRE_BREAK_VALUE,COMMENT_OBS,CALCUL_START_DATE,CALCUL_END_DATE,TIME_FORMAT,BREAKS,COMMENT_TS,COMPILING_ORG,COVERAGE,DATA_COMP,DECIMALS,DISS_ORG,PUBL_ECB,PUBL_MU,PUBL_PUBLIC,TIME_PER_COLLECT,TITLE,TITLE_COMPL,UNIT_INDEX_BASE,UNIT_MEASURE,UNIT_MULT\r\nEST.B.EU000A2X2A25.WT,B,EU000A2X2A25,WT,2021-12-24,-0.58,A,F,,,,,P1D,,,,"ESA 2010 Sectors: S.121, S.122, S.123, S.124, S.125, S.126, S.127, S.128, S.129",,3,,,,,A,Euro sh'

In [701]:
estr = pd.read_csv(StringIO(response_estr.text))
estr
#on l'affiche sous forme de tableau

,KEY,FREQ,BENCHMARK_ITEM,DATA_TYPE_EST,TIME_PERIOD,OBS_VALUE,OBS_STATUS,CONF_STATUS,PRE_BREAK_VALUE,COMMENT_OBS,...,DISS_ORG,PUBL_ECB,PUBL_MU,PUBL_PUBLIC,TIME_PER_COLLECT,TITLE,TITLE_COMPL,UNIT_INDEX_BASE,UNIT_MEASURE,UNIT_MULT
0,EST.B.EU000A2X2A25.WT,B,EU000A2X2A25,WT,2021-12-24,-0.580,A,F,NaN,NaN,...,NaN,NaN,NaN,NaN,A,Euro short-term rate,"Euro short-term rate, Volume-weighted trimmed ...",NaN,PC,0
1,EST.B.EU000A2X2A25.WT,B,EU000A2X2A25,WT,2021-12-27,-0.576,A,F,NaN,NaN,...,NaN,NaN,NaN,NaN,A,Euro short-term rate,"Euro short-term rate, Volume-weighted trimmed ...",NaN,PC,0
2,EST.B.EU000A2X2A25.WT,B,EU000A2X2A25,WT,2021-12-28,-0.575,A,F,NaN,NaN,...,NaN,NaN,NaN,NaN,A,Euro short-term rate,"Euro short-term rate, Volume-weighted trimmed ...",NaN,PC,0
3,EST.B.EU000A2X2A25.WT,B,EU000A2X2A25,WT,2021-12-29,-0.578,A,F,NaN,NaN,...,NaN,NaN,NaN,NaN,A,Euro short-term rate,"Euro short-term rate, Volume-weighted trimmed ...",NaN,PC,0
4,EST.B.EU000A2X2A25.WT,B,EU000A2X2A25,WT,2021-12-30,-0.580,A,F,NaN,NaN,...,NaN,NaN,NaN,NaN,A,Euro short-term rate,"Euro short-term rate, Volume-weighted trimmed ...",NaN,PC,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1204,EST.B.EU000A2X2A25.WT,B,EU000A2X2A25,WT,2026-09-09,2.189,A,F,NaN,NaN,...,NaN,NaN,NaN,NaN,A,Euro short-term rate,"Euro short-term rate, Volume-weighted trimmed ...",NaN,PC,0
1205,EST.B.EU000A2X2A25.WT,B,EU000A2X2A25,WT,2026-09-10,2.190,A,F,NaN,NaN,...,NaN,NaN,NaN,NaN,A,Euro short-term rate,"Euro short-term rate, Volume-weighted trimmed ...",NaN,PC,0
1206,EST.B.EU000A2X2A25.WT,B,EU000A2X2A25,WT,2026-09-11,2.189,A,F,NaN,NaN,...,NaN,NaN,NaN,NaN,A,Euro short-term rate,"Euro short-term rate, Volume-weighted trimmed ...",NaN,PC,0
1207,EST.B.EU000A2X2A25.WT,B,EU000A2X2A25,WT,2026-09-14,2.190,A,F,NaN,NaN,...,NaN,NaN,NaN,NaN,A,Euro short-term rate,"Euro short-term rate, Volume-weighted trimmed ...",NaN,PC,0


In [702]:
estr = estr[["TIME_PERIOD", "OBS_VALUE"]]
estr["TIME_PERIOD"] = pd.to_datetime(estr["TIME_PERIOD"])
#on prends la date et sa valeur

In [703]:
estr["TIME_PERIOD"].dtype

dtype('<M8[us]')

In [704]:
estr = estr.set_index("TIME_PERIOD")

In [705]:
estr.head()

,OBS_VALUE
TIME_PERIOD,
2021-12-24,-0.580
2021-12-27,-0.576
2021-12-28,-0.575
2021-12-29,-0.578
2021-12-30,-0.580


In [706]:
estr["OBS_VALUE"].dtype

dtype('float64')

In [707]:
estr["taux_sans_risque"] = estr["OBS_VALUE"] / 100
estr.head()
#les taux sont en %, on les divise par 100

,OBS_VALUE,taux_sans_risque
TIME_PERIOD,,
2021-12-24,-0.580,-0.00580
2021-12-27,-0.576,-0.00576
2021-12-28,-0.575,-0.00575
2021-12-29,-0.578,-0.00578
2021-12-30,-0.580,-0.00580


In [708]:
estr["taux_sans_risque"] = estr["OBS_VALUE"] / 100

estr = estr.rename(columns={"OBS_VALUE": "taux_rf_en_pourcent"})

estr.head()

,taux_rf_en_pourcent,taux_sans_risque
TIME_PERIOD,,
2021-12-24,-0.580,-0.00580
2021-12-27,-0.576,-0.00576
2021-12-28,-0.575,-0.00575
2021-12-29,-0.578,-0.00578
2021-12-30,-0.580,-0.00580


In [709]:
estr = estr[["taux_sans_risque"]]
#grâce à ça, je viens d'apprendre la différence entre une series et un dataframe
estr.head()

,taux_sans_risque
TIME_PERIOD,
2021-12-24,-0.00580
2021-12-27,-0.00576
2021-12-28,-0.00575
2021-12-29,-0.00578
2021-12-30,-0.00580


In [710]:
estr["taux_sans_risque_daily"] = (1 + estr["taux_sans_risque"]) ** (1/252) - 1
estr["taux_sans_risque_daily"]

TIME_PERIOD
2021-12-24   -0.000023
2021-12-27   -0.000023
2021-12-28   -0.000023
2021-12-29   -0.000023
2021-12-30   -0.000023
                ...   
2026-09-09    0.000086
2026-09-10    0.000086
2026-09-11    0.000086
2026-09-14    0.000086
2026-09-15    0.000086
Name: taux_sans_risque_daily, Length: 1209, dtype: float64

In [711]:
sr_data = pd.concat(
    [candidates_returns, estr],
    axis=1,
    join="inner",
    )
sr_data.head()

#on a concaténé le dataframe des rendements et le dataframe des taux sans risque

,ADP.PA,AI.PA,AIR.PA,AM.PA,ANTIN.PA,CBE.PA,DBG.PA,FRVIA.PA,GFC.PA,GLO.PA,...,ITP.PA,KER.PA,MC.PA,MF.PA,RI.PA,VIE.PA,VIRP.PA,WAVE.PA,taux_sans_risque,taux_sans_risque_daily
2021-12-27,-0.009782,0.012759,0.005910,-0.002647,0.013626,0.0,0.031153,0.006588,0.004951,-0.045964,...,0.005642,0.017467,0.012406,0.014178,-0.002369,-0.004083,0.021661,-0.015326,-0.00576,-0.000023
2021-12-28,0.005388,0.007087,0.009614,0.004246,-0.009351,0.0,-0.004028,0.003636,0.005336,-0.002350,...,0.002805,0.002289,0.006196,-0.010252,0.001899,0.027436,0.001178,0.015564,-0.00575,-0.000023
2021-12-29,0.000447,-0.004822,-0.011462,-0.003171,-0.002360,0.0,0.009606,-0.017391,-0.001225,-0.009423,...,0.012587,0.004139,-0.003968,-0.001883,0.003792,-0.010436,0.016471,0.011494,-0.00578,-0.000023
2021-12-30,0.000893,0.006154,0.004995,0.003181,0.021881,0.0,-0.000501,0.012291,0.006950,0.022592,...,0.015193,0.006112,0.002885,0.001887,0.000944,-0.004963,-0.004630,0.015151,-0.00580,-0.000023
2021-12-31,0.010705,-0.002343,-0.002840,0.004228,-0.001736,0.0,0.017034,0.015784,-0.002030,0.011628,...,0.000000,-0.001271,-0.004109,-0.007533,-0.002358,0.005611,-0.012791,0.014925,-0.00590,-0.000023


In [712]:
#on va calculer le rendement excédentaire, qui correspond au rendement de l'action - le taux sans risque

rendement_excédentaire = candidates_returns.sub(
    sr_data["taux_sans_risque_daily"],
    axis=0,
)
rendement_excédentaire.head()

Ticker,ADP.PA,AI.PA,AIR.PA,AM.PA,ANTIN.PA,CBE.PA,DBG.PA,FRVIA.PA,GFC.PA,GLO.PA,IPS.PA,ITP.PA,KER.PA,MC.PA,MF.PA,RI.PA,VIE.PA,VIRP.PA,WAVE.PA
Date,,,,,,,,,,,,,,,,,,,
2021-12-27,-0.009759,0.012782,0.005933,-0.002624,0.013649,0.000023,0.031176,0.006611,0.004973,-0.045941,-0.001213,0.005665,0.017490,0.012429,0.014201,-0.002346,-0.004060,0.021684,-0.015303
2021-12-28,0.005411,0.007110,0.009637,0.004269,-0.009328,0.000023,-0.004005,0.003659,0.005359,-0.002327,0.004973,0.002828,0.002312,0.006219,-0.010229,0.001922,0.027459,0.001201,0.015587
2021-12-29,0.000470,-0.004799,-0.011439,-0.003148,-0.002337,0.000023,0.009629,-0.017368,-0.001202,-0.009400,0.008644,0.012610,0.004162,-0.003945,-0.001860,0.003815,-0.010413,0.016494,0.011517
2021-12-30,0.000916,0.006177,0.005018,0.003204,0.021904,0.000023,-0.000478,0.012314,0.006973,0.022615,0.014675,0.015216,0.006135,0.002908,0.001910,0.000967,-0.004940,-0.004607,0.015174
2021-12-31,0.010728,-0.002319,-0.002816,0.004252,-0.001713,0.000023,0.017058,0.015808,-0.002006,0.011651,-0.007197,0.000023,-0.001248,-0.004086,-0.007509,-0.002335,0.005634,-0.012767,0.014949


In [713]:
#on passe au calcul du ratio de Sharpe

sharpe_ratio = rendement_excédentaire.mean() / rendement_excédentaire.std() * np.sqrt(252)
sharpe_ratio

Ticker
ADP.PA      0.118528
AI.PA       0.506649
AIR.PA      0.540964
AM.PA       0.911581
ANTIN.PA   -0.508991
CBE.PA     -0.090163
DBG.PA      0.199040
FRVIA.PA   -0.265266
GFC.PA     -0.245171
GLO.PA      0.433980
IPS.PA      0.128181
ITP.PA     -0.126923
KER.PA     -0.402310
MC.PA      -0.233873
MF.PA       0.014702
RI.PA      -0.825893
VIE.PA      0.202743
VIRP.PA    -0.081603
WAVE.PA    -0.120348
dtype: float64

In [744]:
df_rs_et_corr = pd.concat(
    [sharpe_ratio, correlation_20],
    axis=1
    )
df_rs_et_corr.columns = ["Ratio de sharpe", "liste des corrélations"]

In [745]:
df_rs_et_corr = df_rs_et_corr.sort_values("Ratio de sharpe", ascending=False)
df_rs_et_corr

,Ratio de sharpe,liste des corrélations
AM.PA,0.911581,-0.014304
AIR.PA,0.540964,0.375021
AI.PA,0.506649,0.054170
GLO.PA,0.433980,0.404563
VIE.PA,0.202743,-0.002743
DBG.PA,0.199040,0.079876
IPS.PA,0.128181,0.391314
ADP.PA,0.118528,0.056430
MF.PA,0.014702,0.375775
VIRP.PA,-0.081603,0.064587


In [776]:
poids_candidat = 0.05
poids_portefeuille = 1 - poids_candidat

In [777]:
nouveau_portefeuille_AM = (
    poids_portefeuille * portfolio_returns
    + poids_candidat * candidates_returns["AM.PA"]
)
nouveau_portefeuille_AM

Date
2021-12-27    0.012105
2021-12-28    0.006864
2021-12-29   -0.010907
2021-12-30    0.005724
2021-12-31    0.000578
                ...   
2026-09-08    0.018097
2026-09-09   -0.008251
2026-09-10   -0.010520
2026-09-11    0.004223
2026-09-14   -0.019335
Length: 1207, dtype: float64

In [778]:
volatilite_nouveau_AM = nouveau_portefeuille_AM.std()
volatilite_nouveau_AM < portfolio_volatility_direct

np.True_

In [779]:
reduction_vol_AM = (
    portfolio_volatility_direct - volatilite_nouveau_AM
)

reduction_vol_AM

np.float64(0.0005289527968858464)

In [789]:
reductions_volatilite = {}

for ticker in candidates_returns.columns:
    nouveau_portefeuille = (
        poids_candidat * candidates_returns[ticker]
        + poids_portefeuille * portfolio_returns
    )

    reduction = portfolio_volatility_direct - nouveau_portefeuille.std()
    reductions_volatilite[ticker] = reduction

print(reductions_volatilite)

{'ADP.PA': np.float64(0.0005234494384273244), 'AI.PA': np.float64(0.000539672229769016), 'AIR.PA': np.float64(0.0003732731111641737), 'AM.PA': np.float64(0.0005289527968858464), 'ANTIN.PA': np.float64(0.0002873746426414695), 'CBE.PA': np.float64(0.0006553987706075289), 'DBG.PA': np.float64(0.0002322900037476528), 'FRVIA.PA': np.float64(-9.834816505368237e-05), 'GFC.PA': np.float64(0.0005309151902540936), 'GLO.PA': np.float64(0.0004016955707613571), 'IPS.PA': np.float64(0.0004547914126874081), 'ITP.PA': np.float64(0.00044728110007007105), 'KER.PA': np.float64(0.00035826504988089106), 'MC.PA': np.float64(0.0003938826737836486), 'MF.PA': np.float64(0.00041099974583285945), 'RI.PA': np.float64(0.0006046325141009345), 'VIE.PA': np.float64(0.00045564668866275235), 'VIRP.PA': np.float64(0.00048198384460581244), 'WAVE.PA': np.float64(0.0003816543161655264)}


In [790]:
reductions_volatilite[ticker] = portfolio_volatility_direct - nouveau_portefeuille.std()
for ticker, reduction in reductions_volatilite.items():
    print(ticker, reduction)

ADP.PA 0.0005234494384273244
AI.PA 0.000539672229769016
AIR.PA 0.0003732731111641737
AM.PA 0.0005289527968858464
ANTIN.PA 0.0002873746426414695
CBE.PA 0.0006553987706075289
DBG.PA 0.0002322900037476528
FRVIA.PA -9.834816505368237e-05
GFC.PA 0.0005309151902540936
GLO.PA 0.0004016955707613571
IPS.PA 0.0004547914126874081
ITP.PA 0.00044728110007007105
KER.PA 0.00035826504988089106
MC.PA 0.0003938826737836486
MF.PA 0.00041099974583285945
RI.PA 0.0006046325141009345
VIE.PA 0.00045564668866275235
VIRP.PA 0.00048198384460581244
WAVE.PA 0.0003816543161655264


In [792]:
#on va passer le dictionnaire reduction_volatilité en serie grâce à pd.Series

reductions_volatilite = pd.Series(
    reductions_volatilite,
    name="Réduction volatilité"
)
df_rs_et_corr = df_rs_et_corr.join(reductions_volatilite)

In [793]:
df_rs_et_corr

,Ratio de sharpe,liste des corrélations,Réduction volatilité
AM.PA,0.911581,-0.014304,0.000529
AIR.PA,0.540964,0.375021,0.000373
AI.PA,0.506649,0.054170,0.000540
GLO.PA,0.433980,0.404563,0.000402
VIE.PA,0.202743,-0.002743,0.000456
DBG.PA,0.199040,0.079876,0.000232
IPS.PA,0.128181,0.391314,0.000455
ADP.PA,0.118528,0.056430,0.000523
MF.PA,0.014702,0.375775,0.000411
VIRP.PA,-0.081603,0.064587,0.000482


In [796]:
df_rs_et_corr["Réduction vol (%)"] = (
    -df_rs_et_corr["Réduction volatilité"]
    / portfolio_volatility_direct
) * 100
df_rs_et_corr = df_rs_et_corr.drop(columns="Réduction volatilité")
df_rs_et_corr

,Ratio de sharpe,liste des corrélations,Réduction vol (%)
AM.PA,0.911581,-0.014304,-3.453431
AIR.PA,0.540964,0.375021,-2.437028
AI.PA,0.506649,0.054170,-3.523416
GLO.PA,0.433980,0.404563,-2.622593
VIE.PA,0.202743,-0.002743,-2.974830
DBG.PA,0.199040,0.079876,-1.516577
IPS.PA,0.128181,0.391314,-2.969246
ADP.PA,0.118528,0.056430,-3.417501
MF.PA,0.014702,0.375775,-2.683338
VIRP.PA,-0.081603,0.064587,-3.146780


In [797]:
#analyse des résultats: parmi les 118 actions candidates, celle qui avait la corrélation la moins elevée, avec le ratio de Sharpe le plus intéressant 
#ainsi qu'une potentielle baisse de la volatilité assez importante (environ -3.45%) c'est AM.PA a savoir Dassault. C'est en effet une action avec de
#très bon fondamentaux (dette faible, bonne croissance organique, free cash flow important, bonne marges, etc...) et qui correspond énormément à ce 
#portefeuille. 
#J'ai beaucoup aimé faire ce projet, car j'ai réellement mis les mains dans le cambouis, et j'ai essayé d'avoir une méthodologie auquel j'ai essayé
#de me tenir jusqu'à la fin. J'ai fais face à plusieurs difficultés, comme le fait de receuillir les données de la bce via leur API, les nombreuses
#erreurs et approximations dans le rédaction du code qui m'ont couté beaucoup de temps, les données introuvables ou manquantes,etc...
#mais dans l'ensemble j'ai beacoup apprit, et j'ai rendu un résultat assez cohérent, et pour ça, j'en suis fier

In [798]:
print("Merci de m'avoir lu!")

Merci de m'avoir lu!
